In [9]:
# Section 1: Import Required Libraries
import os
from pathlib import Path
import cv2
import torch
import torch.nn.functional as F
import matplotlib.pyplot as plt
from facenet_pytorch import InceptionResnetV1
from torchvision import transforms
from mtcnn import MTCNN


In [10]:
# Section 2: Configure CUDA/CPU Device
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"CUDA available: {torch.cuda.is_available()}")
print(f"Device selected: {device}")

if torch.cuda.is_available():
    print(f"GPU name: {torch.cuda.get_device_name(0)}")
else:
    print("GPU not detected. Running on CPU.")


CUDA available: True
Device selected: cuda
GPU name: NVIDIA GeForce GTX 1650


In [11]:
# Section 3: Initialize Face Recognition Model and Preprocessing Pipeline
resnet = InceptionResnetV1(pretrained='vggface2').eval().to(device)
preprocess = transforms.Compose([
    transforms.ToPILImage(),
    transforms.Resize((160, 160)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.5, 0.5, 0.5], std=[0.5, 0.5, 0.5]),
])


In [13]:
# Section 4: Load Target Image and Convert Color Space
current_dir = Path.cwd()
possible_roots = [current_dir, current_dir.parent, current_dir.parent.parent]

data_root = None
for root in possible_roots:
    candidate = root / 'data'
    if candidate.exists():
        data_root = candidate
        break

if data_root is None:
    raise FileNotFoundError("Could not locate the project data folder from the current notebook working directory.")

target_image_path = data_root / 'test_target3.jpeg'
if not target_image_path.exists():
    raise FileNotFoundError(f"Could not open target image: {target_image_path}")

target_img_cv = cv2.imread(str(target_image_path))
if target_img_cv is None:
    raise ValueError(f"The file exists but OpenCV could not read it: {target_image_path}")

target_img_rgb = cv2.cvtColor(target_img_cv, cv2.COLOR_BGR2RGB)


In [14]:
# Section 5: Detect Faces in the Target Image
detector = MTCNN(steps_threshold=[0.4, 0.5, 0.5])
faces = detector.detect_faces(target_img_rgb)
print(f"Faces found: {len(faces)}")


1/1 ━━━━━━━━━━━━━━━━━━━━ 1s 712ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 278ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 123ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 78ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 55ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 49ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 49ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 44ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 36ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 34ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 35ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 33ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 35ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 34ms/step
50/50 ━━━━━━━━━━━━━━━━━━━━ 0s 6ms/step
3/3 ━━━━━━━━━━━━━━━━━━━━ 0s 65ms/step
Faces found: 5


In [15]:
# Section 6: Crop Detected Faces
cropped_faces = []
for face in faces:
    x, y, width, height = face['box']
    x_start = max(0, x)
    y_start = max(0, y)
    x_end = min(target_img_rgb.shape[1], x_start + width)
    y_end = min(target_img_rgb.shape[0], y_start + height)
    face_crop = target_img_rgb[y_start:y_end, x_start:x_end]
    cropped_faces.append(face_crop)

print(f"Cropped {len(cropped_faces)} face regions.")


Cropped 5 face regions.


In [16]:
# Section 7: Load the Known Reference Face
known_reference_path = data_root / 'rajtilak_face.jpeg'
if not known_reference_path.exists():
    raise FileNotFoundError(f"Could not open reference image: {known_reference_path}")

known_img_cv = cv2.imread(str(known_reference_path))
if known_img_cv is None:
    raise ValueError(f"The file exists but OpenCV could not read it: {known_reference_path}")

known_img_rgb = cv2.cvtColor(known_img_cv, cv2.COLOR_BGR2RGB)


In [17]:
# Section 8: Extract Known Face Embedding
known_tensor = preprocess(known_img_rgb).unsqueeze(0).to(device)
with torch.no_grad():
    known_embedding = resnet(known_tensor).cpu()


In [18]:
# Section 9: Extract Embeddings for Each Detected Face
face_embeddings = []
for i, face_matrix in enumerate(cropped_faces):
    unknown_tensor = preprocess(face_matrix).unsqueeze(0).to(device)
    with torch.no_grad():
        unknown_embedding = resnet(unknown_tensor).cpu()
    face_embeddings.append(unknown_embedding)
    print(f"Embedding extracted for detected face {i + 1}")


Embedding extracted for detected face 1
Embedding extracted for detected face 2
Embedding extracted for detected face 3
Embedding extracted for detected face 4
Embedding extracted for detected face 5


In [21]:
# Section 10: Compute Cosine Similarity and Match Threshold
MATCH_THRESHOLD = 0.60
confirmed_matches = 0


In [20]:
# Section 11: Print Match Results
for i, embedding in enumerate(face_embeddings):
    similarity = F.cosine_similarity(known_embedding, embedding).item()
    is_match = similarity > MATCH_THRESHOLD
    if is_match:
        confirmed_matches += 1
        status = "MATCH"
    else:
        status = "NO MATCH"
    print(f"Target {i + 1}: similarity = {similarity:.4f} -> {status}")

print(f"Confirmed matches found: {confirmed_matches}")


Target 1: similarity = 0.4963 -> NO MATCH
Target 2: similarity = 0.3944 -> NO MATCH
Target 3: similarity = 0.3023 -> NO MATCH
Target 4: similarity = 0.3111 -> NO MATCH
Target 5: similarity = 0.2034 -> NO MATCH
Confirmed matches found: 0
